# Feature Engineering for MarketPredictor

This notebook aims to create and analyze advanced explanatory variables (features) from stock market data (CAC40, S&P500).

## 1. Import libraries and data
Import pandas, numpy, and load the cleaned data from the previous notebook or CSV files.

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf

In [ ]:
# Load previously saved data
df_sp500 = pd.read_csv('../data/raw/sp500.csv', parse_dates=['Date'])
df_cac40 = pd.read_csv('../data/raw/cac40.csv', parse_dates=['Date'])

In [ ]:
# Check types and first rows
print(df_sp500.dtypes)
print(df_sp500.head(10))

# Force 'Close' column to numeric, replace errors with NaN
df_sp500['Close'] = pd.to_numeric(df_sp500['Close'], errors='coerce')
df_cac40['Close'] = pd.to_numeric(df_cac40['Close'], errors='coerce')

Date      datetime64[ns]
Close             object
High              object
Low               object
Open              object
Volume            object
dtype: object
        Date               Close                High                 Low  \
0        NaT               ^GSPC               ^GSPC               ^GSPC   
1 2015-01-02   2058.199951171875   2072.360107421875     2046.0400390625   
2 2015-01-05  2020.5799560546875    2054.43994140625  2017.3399658203125   
3 2015-01-06  2002.6099853515625             2030.25    1992.43994140625   
4 2015-01-07  2025.9000244140625  2029.6099853515625   2005.550048828125   
5 2015-01-08   2062.139892578125      2064.080078125  2030.6099853515625   
6 2015-01-09    2044.81005859375   2064.429931640625  2038.3299560546875   
7 2015-01-12   2028.260009765625   2049.300048828125  2022.5799560546875   
8 2015-01-13   2023.030029296875   2056.929931640625             2008.25   
9 2015-01-14    2011.27001953125  2018.4000244140625    1988.43994140625   


In [ ]:
# Remove the first unwanted row (tickers)
df_sp500 = df_sp500.iloc[1:].reset_index(drop=True)
df_cac40 = df_cac40.iloc[1:].reset_index(drop=True)

# Force all numeric columns to float
for col in ['Close', 'High', 'Low', 'Open', 'Volume']:
    df_sp500[col] = pd.to_numeric(df_sp500[col], errors='coerce')
    df_cac40[col] = pd.to_numeric(df_cac40[col], errors='coerce')

In [5]:
df_cac40.head()

,Date,Close,High,Low,Open,Volume
0,2015-01-02,4252.290039,4311.000000,4224.339844,4294.049805,69809300
1,2015-01-05,4111.359863,4276.919922,4105.450195,4221.990234,137887700
2,2015-01-06,4083.500000,4151.410156,4076.159912,4129.890137,130814400
3,2015-01-07,4112.729980,4144.950195,4080.780029,4111.729980,121316600
4,2015-01-08,4260.189941,4270.109863,4163.629883,4176.160156,154417100


In [6]:
df_sp500.head()

,Date,Close,High,Low,Open,Volume
0,2015-01-02,2058.199951,2072.360107,2046.040039,2058.899902,2708700000
1,2015-01-05,2020.579956,2054.439941,2017.339966,2054.439941,3799120000
2,2015-01-06,2002.609985,2030.250000,1992.439941,2022.150024,4460110000
3,2015-01-07,2025.900024,2029.609985,2005.550049,2005.550049,3805480000
4,2015-01-08,2062.139893,2064.080078,2030.609985,2030.609985,3934010000


## 2. Creation of advanced features
 
### Addition of sector features (US sector ETFs, exogenous variables)
Here we add the daily variation of several US sector ETFs as exogenous features.
 
We will create technical indicators and features based on sliding windows to capture market dynamics.

In [ ]:
# Robust addition of US sector features with diagnostics and correct MultiIndex extraction
sector_etfs = ['XLF', 'XLK', 'XLE', 'XLY', 'XLI', 'XLV', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
start, end = df_sp500['Date'].min().strftime('%Y-%m-%d'), df_sp500['Date'].max().strftime('%Y-%m-%d')
sector_raw = yf.download(sector_etfs, start=start, end=end)
print('Downloaded DataFrame columns:', sector_raw.columns)
print('DataFrame preview:')
display(sector_raw.head())
 
# Extract closing prices via the 'Close' level of the MultiIndex
if isinstance(sector_raw.columns, pd.MultiIndex) and 'Close' in sector_raw.columns.get_level_values(0):
    close_prices = sector_raw.xs('Close', axis=1, level=0)
    sector_data = close_prices.pct_change().dropna()
    sector_data.columns = [f'sector_{col}_ret' for col in sector_data.columns]
    df_sp500 = df_sp500.merge(sector_data, left_on='Date', right_index=True, how='left')
    print('Sector features added successfully.')
else:
    print('Unable to find closing prices in sector data. Check DataFrame structure.')

[*********************100%***********************]  11 of 11 completed

Colonnes du DataFrame téléchargé : MultiIndex([( 'Close',  'XLB'),
            ( 'Close',  'XLC'),
            ( 'Close',  'XLE'),
            ( 'Close',  'XLF'),
            ( 'Close',  'XLI'),
            ( 'Close',  'XLK'),
            ( 'Close',  'XLP'),
            ( 'Close', 'XLRE'),
            ( 'Close',  'XLU'),
            ( 'Close',  'XLV'),
            ( 'Close',  'XLY'),
            (  'High',  'XLB'),
            (  'High',  'XLC'),
            (  'High',  'XLE'),
            (  'High',  'XLF'),
            (  'High',  'XLI'),
            (  'High',  'XLK'),
            (  'High',  'XLP'),
            (  'High', 'XLRE'),
            (  'High',  'XLU'),
            (  'High',  'XLV'),
            (  'High',  'XLY'),
            (   'Low',  'XLB'),
            (   'Low',  'XLC'),
            (   'Low',  'XLE'),
            (   'Low',  'XLF'),
            (   'Low',  'XLI'),
            (   'Low',  'XLK'),
            (   'Low',  'XLP'),
            (   'Low', 'XLRE'),
     

Price           Close                                                  \
Ticker            XLB XLC        XLE        XLF        XLI        XLK   
Date                                                                    
2015-01-02  19.495251 NaN  25.714987  16.358452  46.341240  18.106766   
2015-01-05  18.998344 NaN  24.651203  16.014479  45.258781  17.830357   
2015-01-06  18.830050 NaN  24.289070  15.769728  44.701138  17.615376   
2015-01-07  19.042425 NaN  24.340811  15.935106  45.037361  17.764544   
2015-01-08  19.495251 NaN  24.887241  16.173227  45.939434  18.155022   

Price                                             ... Volume            \
Ticker            XLP XLRE        XLU        XLV  ...    XLC       XLE   
Date                                              ...                    
2015-01-02  36.067268  NaN  16.626711  57.292412  ...    NaN  55498200   
2015-01-05  35.813633  NaN  16.423433  57.000225  ...    NaN  90790400   
2015-01-06  35.768894  NaN  16.433945  56.808231  ...    NaN  83753200   
2015-01-07  36.380573  NaN  16.595169  58.143890  ...    NaN  62384800   
2015-01-08  36.932575  NaN  16.710817  59.137302  ...    NaN  56373800   

Price                                                                        \
Ticker           XLF       XLI       XLK       XLP XLRE       XLU       XLV   
Date                                                                          
2015-01-02  40511471  10982800  16990400   7612100  NaN  33829400   7516800   
2015-01-05  50770502  15144700  14389400   8272700  NaN  47823600  10784800   
2015-01-06  57454463  19209800  17491200  11112400  NaN  38787400  13344500   
2015-01-07  36287049  11770300  13069200   9226100  NaN  28644600  14146400   
2015-01-08  37995923  11419800  28423400  12904100  NaN  28055400  22138400   

Price                 
Ticker           XLY  
Date                  
2015-01-02  12371000  
2015-01-05  17165800  
2015-01-06  14346000  
2015-01-07  14822000  
2015-01-08  12798200  

[5 rows x 55 columns]

Features sectorielles ajoutées avec succès.


In [ ]:
# Robust addition of sector features for CAC40 (MultiIndex extraction)
sector_etfs_cac = ['EXX', 'EXP']
start, end = df_cac40['Date'].min().strftime('%Y-%m-%d'), df_cac40['Date'].max().strftime('%Y-%m-%d')
sector_raw_cac = yf.download(sector_etfs_cac, start=start, end=end)
print('Downloaded DataFrame columns (CAC40):', sector_raw_cac.columns)
print('DataFrame preview (CAC40):')
display(sector_raw_cac.head())
 
# Extract closing prices via the 'Close' level of the MultiIndex
if isinstance(sector_raw_cac.columns, pd.MultiIndex) and 'Close' in sector_raw_cac.columns.get_level_values(0):
    close_prices_cac = sector_raw_cac.xs('Close', axis=1, level=0)
    sector_data_cac = close_prices_cac.pct_change().dropna()
    sector_data_cac.columns = [f'sector_{col}_ret' for col in sector_data_cac.columns]
    df_cac40 = df_cac40.merge(sector_data_cac, left_on='Date', right_index=True, how='left')
    print('CAC40 sector features added successfully.')
else:
    print('Unable to find closing prices in CAC40 sector data. Check DataFrame structure.')

[*********************100%***********************]  2 of 2 completed

Colonnes du DataFrame téléchargé (CAC40) : MultiIndex([( 'Close', 'EXP'),
            ( 'Close', 'EXX'),
            (  'High', 'EXP'),
            (  'High', 'EXX'),
            (   'Low', 'EXP'),
            (   'Low', 'EXX'),
            (  'Open', 'EXP'),
            (  'Open', 'EXX'),
            ('Volume', 'EXP'),
            ('Volume', 'EXX')],
           names=['Price', 'Ticker'])
Aperçu du DataFrame (CAC40) :


Price           Close                   High                    Low  \
Ticker            EXP         EXX        EXP         EXX        EXP   
Date                                                                  
2015-01-02  73.070419  105.139999  74.331564  105.570000  71.828243   
2015-01-05  69.182671  101.669998  72.093728  106.669998  68.585290   
2015-01-06  67.997406    0.155000  69.599913    0.155000  67.371579   
2015-01-07  69.030952  103.209999  70.111931  103.620003  68.310298   
2015-01-08  71.629112  104.989998  72.245462  105.449997  69.391295   

Price                        Open               Volume            
Ticker             EXX        EXP         EXX      EXP       EXX  
Date                                                              
2015-01-02  102.839996  72.899738  103.750000   470600   63377.0  
2015-01-05  101.620003  71.676506  103.160004  1052700  342032.0  
2015-01-06    0.155000  69.391302    0.155000   986500     820.0  
2015-01-07  101.699997  68.860271  103.419998   664600  174010.0  
2015-01-08  101.599998  69.467155  104.870003  1102100  355703.0

Features sectorielles CAC40 ajoutées avec succès.


C:\Users\msagh\AppData\Local\Temp\ipykernel_24956\2604740543.py:12: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  sector_data_cac = close_prices_cac.pct_change().dropna()


### 2.1 Moving averages and ratios
Simple and exponential moving averages are used to smooth prices and detect trends. Ratios allow comparison of the current price to these averages.

In [11]:
def add_moving_averages(df, windows=[5, 10, 20, 50, 100, 200]):
    for w in windows:
        df[f'MA{w}'] = df['Close'].rolling(window=w).mean()
        df[f'EMA{w}'] = df['Close'].ewm(span=w, adjust=False).mean()
        df[f'Close/MA{w}'] = df['Close'] / df[f'MA{w}']
    return df

df_sp500 = add_moving_averages(df_sp500)
df_cac40 = add_moving_averages(df_cac40)

### 2.2 Momentum and trend indicators
RSI, MACD, and momentum are classic indicators to detect the strength and direction of a trend.

In [12]:
def add_rsi(df, window=14):
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    return df

def add_macd(df, span_short=12, span_long=26, span_signal=9):
    ema_short = df['Close'].ewm(span=span_short, adjust=False).mean()
    ema_long = df['Close'].ewm(span=span_long, adjust=False).mean()
    df['MACD'] = ema_short - ema_long
    df['MACD_signal'] = df['MACD'].ewm(span=span_signal, adjust=False).mean()
    return df

def add_momentum(df, window=10):
    df['Momentum'] = df['Close'] - df['Close'].shift(window)
    return df

df_sp500 = add_rsi(df_sp500)
df_sp500 = add_macd(df_sp500)
df_sp500 = add_momentum(df_sp500)
df_cac40 = add_rsi(df_cac40)
df_cac40 = add_macd(df_cac40)
df_cac40 = add_momentum(df_cac40)

### 2.3 Volatility and dispersion features
Volatility (rolling standard deviation) and the amplitude of variations are indicators of risk and uncertainty.

In [13]:
def add_volatility(df, windows=[5, 10, 20]):
    for w in windows:
        df[f'Volatility_{w}'] = df['Close'].rolling(window=w).std()
        df[f'Range_{w}'] = (df['High'] - df['Low']).rolling(window=w).mean()
    return df

df_sp500 = add_volatility(df_sp500)
df_cac40 = add_volatility(df_cac40)

### 2.4 Return and variation features
Returns and their cumulative or differential versions are essential for directional modeling.

In [14]:
def add_returns(df, windows=[1, 5, 10]):
    df['Return_1d'] = df['Close'].pct_change(1)
    for w in windows:
        df[f'Return_{w}d'] = df['Close'].pct_change(w)
        df[f'Return_{w}d_future'] = df['Close'].pct_change(-w)  # Pour la cible future
    return df

df_sp500 = add_returns(df_sp500)
df_cac40 = add_returns(df_cac40)

### 2.5 Calendar and temporal features
Day of the week, day of the month, month, etc. can influence market behavior.

In [15]:
def add_calendar_features(df):
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['Day'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
    return df

df_sp500 = add_calendar_features(df_sp500)
df_cac40 = add_calendar_features(df_cac40)

### 2.6 Target variable: next-day direction
We create a binary variable indicating whether the closing price the next day is higher than today (1: up, 0: down or stable).

In [16]:
def add_target_direction(df):
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    return df

df_sp500 = add_target_direction(df_sp500)
df_cac40 = add_target_direction(df_cac40)

## 3. Contrôle de qualité et sauvegarde

In [ ]:
# Display the first rows for quality control
print(df_sp500.head())
print(df_cac40.head())
 
# Check for NaN and document features
print('Number of NaN per column (SP500):')
print(df_sp500.isna().sum().sort_values(ascending=False))
print('List of created features (SP500):')
print(list(df_sp500.columns))
print('Number of NaN per column (CAC40):')
print(df_cac40.isna().sum().sort_values(ascending=False))
print('List of created features (CAC40):')
print(list(df_cac40.columns))
 
# Save the enriched datasets and verify writing
import os
sp500_path = '../data/processed/sp500_features.csv'
cac40_path = '../data/processed/cac40_features.csv'
df_sp500.to_csv(sp500_path, index=False)
df_cac40.to_csv(cac40_path, index=False)
print(f"SP500 file saved: {os.path.abspath(sp500_path)} | Exists: {os.path.exists(sp500_path)}")
print(f"CAC40 file saved: {os.path.abspath(cac40_path)} | Exists: {os.path.exists(cac40_path)}")

        Date        Close         High          Low         Open      Volume  \
0 2015-01-02  2058.199951  2072.360107  2046.040039  2058.899902  2708700000   
1 2015-01-05  2020.579956  2054.439941  2017.339966  2054.439941  3799120000   
2 2015-01-06  2002.609985  2030.250000  1992.439941  2022.150024  4460110000   
3 2015-01-07  2025.900024  2029.609985  2005.550049  2005.550049  3805480000   
4 2015-01-08  2062.139893  2064.080078  2030.609985  2030.609985  3934010000   

   sector_XLB_ret  sector_XLC_ret  sector_XLE_ret  sector_XLF_ret  ...  \
0             NaN             NaN             NaN             NaN  ...   
1             NaN             NaN             NaN             NaN  ...   
2             NaN             NaN             NaN             NaN  ...   
3             NaN             NaN             NaN             NaN  ...   
4             NaN             NaN             NaN             NaN  ...   

   Return_5d_future  Return_10d  Return_10d_future  DayOfWeek  Day  Month 